## LRModelQ3

In [1]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

### Data Preparation

In [2]:
# --- Load Clean Data (Yelp 3-class sentiment) ---
df = pd.read_csv(DATA_FILE)

In [3]:
# --- Split into train/test (same split as Q2 so results stay comparable) ---
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

### Hyperparameter Search

In [4]:
# --- Base pipeline to tune (same as Q2) ---
# TF-IDF settings left at defaults because they are searched below.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(strip_accents="unicode")),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

In [5]:
# --- Hyperparameter search space ---
# Same TF-IDF space as the other 3 models so the comparison stays fair. 360 combinations.
param_dist = {
    'tfidf__max_features': [10000, 20000, 30000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2, 3],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [0.1, 0.5, 1.0, 2.0, 5.0],
    'clf__class_weight': [None, 'balanced']
}

In [6]:
# --- Randomized search ---
# 10 combinations x 5 folds = 50 fits. f1_macro not accuracy, since neutral is the small class.
search = RandomizedSearchCV(
    pipeline, param_dist,
    n_iter=10, cv=5,
    scoring='f1_macro',
    random_state=42, n_jobs=-1, verbose=0
)

search.fit(X_train, y_train);

### Search Results

In [7]:
# --- Report best hyperparameters and score ---
print("=== Q3: Best Hyperparameters ===")
print("=" * 45)
for param, value in search.best_params_.items():
    name = param.replace('clf__', '').replace('tfidf__', '')
    print(f"  {name:<20} : {value}")
print("=" * 45)
print("Best CV Macro F1:", search.best_score_)

=== Q3: Best Hyperparameters ===
  sublinear_tf         : True
  ngram_range          : (1, 1)
  min_df               : 2
  max_features         : 20000
  class_weight         : balanced
  C                    : 0.5
Best CV Macro F1: 0.699672785723403


### Save Tuned Model

In [8]:
# --- Save tuned model for Q4 ---
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(search.best_estimator_, MODEL_DIR / 'lr_tuned.joblib')

['C:\\Users\\yingx\\Desktop\\TextAssignment\\Part_B\\models\\lr_tuned.joblib']